In [ ]:
import sys
from pathlib import Path
import duckdb

sys.path.insert(0, str(Path.cwd().parent))

In [1]:
from paths import BBO_GLOB

In [2]:
con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")

Two-sided quotes span **2023-05-02** to **2026-06-26**.

In [6]:
con.sql(f"""
    SELECT
        MIN(CAST(COALESCE(ts_event, ts_recv) AS DATE)) AS earliest,
        MAX(CAST(COALESCE(ts_event, ts_recv) AS DATE)) AS latest
    FROM read_parquet('{BBO_GLOB}')
    WHERE (symbol LIKE '%_OMC%' OR symbol LIKE '%_OMP%')
      AND side = 'N'
      AND flags IN (128, 160)
      AND bid_px_00 > 0 AND ask_px_00 > 0
""").df()

,earliest,latest
0,2023-05-02,2026-06-26


Two-sided quotes per year: 
- 2,485 (2023)
- 2,069 (2024)
- 25,969,129 (2025) 
- 27,749,569 (2026).

In [5]:
con.sql(f"""
    SELECT
        YEAR(COALESCE(ts_event, ts_recv)) AS year,
        COUNT(*)                          AS n_quotes
    FROM read_parquet('{BBO_GLOB}')
    WHERE (symbol LIKE '%_OMC%' OR symbol LIKE '%_OMP%')
      AND side = 'N'
      AND flags IN (128, 160)
      AND bid_px_00 > 0 AND ask_px_00 > 0
    GROUP BY 1
    ORDER BY 1
""").df()

,year,n_quotes
0,2023,2485
1,2024,2069
2,2025,25969129
3,2026,27749569
